In [5]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

ModuleNotFoundError: No module named 'sklearn'

In [6]:
import yaml
with open('../config.yaml', 'r') as file:
    config = yaml.safe_load(file)

Opening both files using "yaml"

In [29]:
from pathlib import Path
import pandas as pd

base_path = Path("../config.yaml").parent
red_wines = pd.read_csv(base_path / config["input_data"]["file1"])
red_wines.head()

,version https://git-lfs.github.com/spec/v1
0,oid sha256:02906b019d098855b97c90751fbca950c27...
1,size 719879


In [8]:
red_wines.info()

<class 'pandas.DataFrame'>
RangeIndex: 2 entries, 0 to 1
Data columns (total 1 columns):
 #   Column                                      Non-Null Count  Dtype
---  ------                                      --------------  -----
 0   version https://git-lfs.github.com/spec/v1  2 non-null      str  
dtypes: str(1)
memory usage: 148.0 bytes


In [9]:
wine_varieties = pd.read_csv(base_path / config["input_data"]["file2"])
wine_varieties.head()

,version https://git-lfs.github.com/spec/v1
0,oid sha256:8f04e6443dba739b916a1bc09a7be0558f7...
1,size 20977


Extracting Variety out of Wine Names so it can be merged with Varieties.csv

In [10]:
common_varieties = wine_varieties["Variety"].unique()

def find_variety(name):
    for v in common_varieties:
        if v in name:
            return v
    return "Other"

red_wines["variety"] = red_wines["Name"].apply(find_variety)
red_wines.head()

KeyError: 'Variety'

Checking for null values

In [11]:
print(red_wines.isnull().sum())
print(wine_varieties.isnull().sum())
print(red_wines.isnull().mean() * 100)  # as percentages
print(wine_varieties.isnull().mean() * 100)  # as percentages

version https://git-lfs.github.com/spec/v1    0
dtype: int64
version https://git-lfs.github.com/spec/v1    0
dtype: int64
version https://git-lfs.github.com/spec/v1    0.0
dtype: float64
version https://git-lfs.github.com/spec/v1    0.0
dtype: float64


Counting Varieties in red_wines and understanding how many "others" there are. Merging might not be a good decision and "varieties" not needed.

In [11]:
red_wines["variety"].value_counts()

variety
Other                        5550
Cabernet Sauvignon            549
Merlot                        269
Pinot Noir                    260
Shiraz                        231
Brunello                      172
Syrah                         164
Malbec                        161
Primitivo                     120
Barbera                       119
Tempranillo                   110
Montepulciano                 107
Pinotage                       76
Nero d'Avola                   62
Zinfandel                      59
Zweigelt                       51
Blaufränkisch                  47
Negroamaro                     40
Lagrein                        39
Nebbiolo                       38
Sangiovese                     38
Dolcetto                       32
Aglianico                      31
Cabernet Franc                 30
Grenache                       24
Teroldego                      19
Carignan                       14
Tannat                         12
Susumaniello                   12
Corvin

Merging datasets

In [ ]:
# red.to_csv("../data/raw/red_ml_ready.csv", index=False)

In [45]:
print(red["variety_ml"].unique())

<StringArray>
[    'Bordeaux Blend',        'Rhone Blend',        'Generic Red',
            'Corvina',         'Pinot Noir',              'Blend',
         'Sangiovese',            'Unknown',          'Primitivo',
             'Shiraz', 'Cabernet Sauvignon',          'Other Red',
        'Tempranillo',            'Barbera',              'Syrah',
           'Grenache',             'Merlot',           'Nebbiolo',
             'Malbec',      'Montepulciano']
Length: 20, dtype: str


In [46]:
red["variety_ml"] = red["variety_ml"].replace({
    "Other Red": "Rare Varieties",
    "Generic Red": "Unspecified Red",
    "Unknown": "Unknown Variety"
})

In [47]:
print(red["variety_ml"].value_counts())

variety_ml
Unknown Variety       1390
Bordeaux Blend        1216
Rare Varieties         889
Tempranillo            841
Unspecified Red        732
Pinot Noir             479
Cabernet Sauvignon     478
Sangiovese             447
Rhone Blend            304
Nebbiolo               271
Merlot                 259
Corvina                240
Shiraz                 219
Syrah                  206
Malbec                 178
Primitivo              129
Barbera                127
Montepulciano          113
Grenache                75
Blend                   73
Name: count, dtype: int64


In [49]:
RENAME_MAP = {
    "Other Red": "Rare Varieties",
    "Generic Red": "Unspecified Red",
    "Unknown": "Unknown Variety",

    # Optional consistency cleanup
    "Bordeaux Blend": "Bordeaux Blend",
    "Rhone Blend": "Rhône Blend",
    "Cabernet Sauvignon": "Cabernet Sauvignon",
    "Pinot Noir": "Pinot Noir",
    "Merlot": "Merlot",
    "Tempranillo": "Tempranillo",
    "Sangiovese": "Sangiovese",
    "Nebbiolo": "Nebbiolo",
    "Syrah": "Syrah",
    "Shiraz": "Shiraz",
    "Malbec": "Malbec",
    "Grenache": "Grenache",
    "Corvina": "Corvina",
    "Barbera": "Barbera",
    "Montepulciano": "Montepulciano",
    "Zinfandel": "Zinfandel",
    "Carmenere": "Carménère",
    "Nero D Avola": "Nero d'Avola"
}

red["variety_ml"] = red["variety_ml"].replace(RENAME_MAP)

In [50]:
print(red["variety_ml"].value_counts())

variety_ml
Unknown Variety       1390
Bordeaux Blend        1216
Rare Varieties         889
Tempranillo            841
Unspecified Red        732
Pinot Noir             479
Cabernet Sauvignon     478
Sangiovese             447
Rhône Blend            304
Nebbiolo               271
Merlot                 259
Corvina                240
Shiraz                 219
Syrah                  206
Malbec                 178
Primitivo              129
Barbera                127
Montepulciano          113
Grenache                75
Blend                   73
Name: count, dtype: int64


In [51]:
print(red[["name", "region", "variety_ml"]].head(20))

                                                name                   region  \
0                                       Pomerol 2011                  Pomerol   
1                                         Lirac 2017                    Lirac   
2                 Erta e China Rosso di Toscana 2015                  Toscana   
3                                     Bardolino 2019                Bardolino   
4                     Ried Scheibner Pinot Noir 2016                Carnuntum   
5                   Gigondas (Nobles Terrasses) 2017                 Gigondas   
6                  Marion's Vineyard Pinot Noir 2016                Wairarapa   
7                                     Red Blend 2014             Itata Valley   
8                                       Chianti 2015                  Chianti   
9                                     Tradition 2014                Minervois   
10                              Chianti Riserva 2013                  Chianti   
11                          

In [52]:
red.to_csv("../data/raw/red_ml_final.csv", index=False)

In [4]:
# Minimal dataset for modeling only
ml_df = red[red[
    "country",
    "region",
    "price",
    "rating",
    "variety_ml"
]]

ml_df.to_csv("../data/raw/red_ml_model_input.csv", index=False)

NameError: name 'red' is not defined

In [56]:
print(red["variety_ml"].unique())

<StringArray>
[    'Bordeaux Blend',        'Rhône Blend',    'Unspecified Red',
            'Corvina',         'Pinot Noir',              'Blend',
         'Sangiovese',    'Unknown Variety',          'Primitivo',
             'Shiraz', 'Cabernet Sauvignon',     'Rare Varieties',
        'Tempranillo',            'Barbera',              'Syrah',
           'Grenache',             'Merlot',           'Nebbiolo',
             'Malbec',      'Montepulciano']
Length: 20, dtype: str


In [2]:
ml_df

NameError: name 'ml_df' is not defined